In [1]:
#CELL 1 — Imports + Seed
import os, json, random
import numpy as np
import pandas as pd

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("✅ Seed set:", SEED)

✅ Seed set: 42


In [2]:
#CELL 2 — Load Data
train_path = "site_1_train_data.csv"
test_path  = "site_1_unseen_input_data.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print("✅ Loaded")
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain columns:", list(train_df.columns))
print("\nTest columns :", list(test_df.columns))

✅ Loaded
Train shape: (25081, 16)
Test shape : (10872, 14)

Train columns: ['year', 'month', 'day', 'hour', 'O3_forecast', 'NO2_forecast', 'T_forecast', 'q_forecast', 'u_forecast', 'v_forecast', 'w_forecast', 'NO2_satellite', 'HCHO_satellite', 'ratio_satellite', 'O3_target', 'NO2_target']

Test columns : ['year', 'month', 'day', 'hour', 'O3_forecast', 'NO2_forecast', 'T_forecast', 'q_forecast', 'u_forecast', 'v_forecast', 'w_forecast', 'NO2_satellite', 'HCHO_satellite', 'ratio_satellite']


In [3]:
#CELL 3 — Build datetime + sort
train_df["datetime"] = pd.to_datetime(train_df[["year","month","day","hour"]])
test_df["datetime"]  = pd.to_datetime(test_df[["year","month","day","hour"]])

train_df = train_df.sort_values("datetime").reset_index(drop=True)
test_df  = test_df.sort_values("datetime").reset_index(drop=True)

print("✅ Datetime built + sorted")
print(train_df[["datetime"]].head())
print(train_df[["datetime"]].tail())

✅ Datetime built + sorted
             datetime
0 2019-07-14 00:00:00
1 2019-07-14 01:00:00
2 2019-07-14 02:00:00
3 2019-07-14 03:00:00
4 2019-07-14 04:00:00
                 datetime
25076 2024-06-28 20:00:00
25077 2024-06-28 21:00:00
25078 2024-06-28 22:00:00
25079 2024-06-28 23:00:00
25080 2024-06-30 00:00:00


In [4]:
#CELL 4 — Hourly reindex (critical)
train_df = train_df.set_index("datetime").sort_index()
test_df  = test_df.set_index("datetime").sort_index()

train_idx = pd.date_range(train_df.index.min(), train_df.index.max(), freq="h")
test_idx  = pd.date_range(test_df.index.min(),  test_df.index.max(),  freq="h")

train_df = train_df.reindex(train_idx)
test_df  = test_df.reindex(test_idx)

# Recreate time columns after reindex
for df in [train_df, test_df]:
    df["year"]  = df.index.year
    df["month"] = df.index.month
    df["day"]   = df.index.day
    df["hour"]  = df.index.hour

print("✅ Reindexed to hourly grid")
print("Train missing cells:", train_df.isna().sum().sum())
print("Test  missing cells:", test_df.isna().sum().sum())

✅ Reindexed to hourly grid
Train missing cells: 293905
Test  missing cells: 355253


In [5]:
#CELL 5 — Missing flags + fill (time-series safe)
def add_missing_flags(df, cols):
    for c in cols:
        df[f"{c}__is_missing"] = df[c].isna().astype(int)
    return df

numeric_cols_train = train_df.select_dtypes(include=np.number).columns
numeric_cols_test  = test_df.select_dtypes(include=np.number).columns
common_numeric = numeric_cols_train.intersection(numeric_cols_test)

train_df = add_missing_flags(train_df, common_numeric)
test_df  = add_missing_flags(test_df, common_numeric)

# Fill numeric columns
train_df[common_numeric] = train_df[common_numeric].replace([np.inf, -np.inf], np.nan)
test_df[common_numeric]  = test_df[common_numeric].replace([np.inf, -np.inf], np.nan)

train_df[common_numeric] = train_df[common_numeric].interpolate(method="time").ffill().bfill()
test_df[common_numeric]  = test_df[common_numeric].interpolate(method="time").ffill().bfill()

print("✅ Missing handled")
print("Remaining NaNs train:", train_df.isna().sum().sum())
print("Remaining NaNs test :", test_df.isna().sum().sum())

✅ Missing handled
Remaining NaNs train: 36864
Remaining NaNs test : 0


In [6]:
#CELL 6 — Feature engineering (cyclical + day-of-week + wind)
# Cyclical encoding + day-of-week
for df in [train_df, test_df]:
    df["hour_sin"]  = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"]  = np.cos(2*np.pi*df["hour"]/24)
    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

    df["dow"]     = df.index.dayofweek
    df["dow_sin"] = np.sin(2*np.pi*df["dow"]/7)
    df["dow_cos"] = np.cos(2*np.pi*df["dow"]/7)

# Wind features
def add_wind_features(df):
    if {"u_forecast","v_forecast"}.issubset(df.columns):
        df["wind_speed_h"] = np.sqrt(df["u_forecast"]**2 + df["v_forecast"]**2)
        df["wind_dir_rad"] = np.arctan2(df["v_forecast"], df["u_forecast"])
        df["wind_dir_sin"] = np.sin(df["wind_dir_rad"])
        df["wind_dir_cos"] = np.cos(df["wind_dir_rad"])
    return df

train_df = add_wind_features(train_df)
test_df  = add_wind_features(test_df)

print("✅ Time + wind features added")

✅ Time + wind features added


In [7]:
#CELL 7 — Lag + rolling features (key drivers)
def add_lags(df, cols, lags=(1,2,3,6,12,24)):
    for c in cols:
        for L in lags:
            df[f"{c}__lag{L}"] = df[c].shift(L)
    return df

def add_roll_stats(df, cols, windows=(3,6,12,24)):
    for c in cols:
        for w in windows:
            df[f"{c}__rollmean{w}"] = df[c].rolling(w, min_periods=1).mean()
            df[f"{c}__rollstd{w}"]  = df[c].rolling(w, min_periods=1).std().fillna(0.0)
    return df

key_cols = [c for c in [
    "O3_forecast","NO2_forecast",
    "T_forecast","q_forecast","u_forecast","v_forecast","w_forecast",
    #"NO2_satellite","HCHO_satellite","ratio_satellite"
] if c in train_df.columns]

train_df = add_lags(train_df, key_cols)
test_df  = add_lags(test_df, key_cols)

lag_cols = [c for c in train_df.columns if "__lag" in c]
train_df[lag_cols] = train_df[lag_cols].ffill().bfill()
test_df[lag_cols]  = test_df[lag_cols].ffill().bfill()

train_df = add_roll_stats(train_df, key_cols)
test_df  = add_roll_stats(test_df, key_cols)

print("✅ Lag + rolling features added")
print("Feature count now:", train_df.shape[1])

✅ Lag + rolling features added
Feature count now: 139


/var/folders/1_/lbw8c9m97nz0sfdrrsz64n1h0000gn/T/ipykernel_95901/141444476.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{c}__rollstd{w}"]  = df[c].rolling(w, min_periods=1).std().fillna(0.0)
/var/folders/1_/lbw8c9m97nz0sfdrrsz64n1h0000gn/T/ipykernel_95901/141444476.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{c}__rollmean{w}"] = df[c].rolling(w, min_periods=1).mean()


In [8]:
#CELL 8 — Split X / y (targets only exist in train)
target_cols = [c for c in train_df.columns if "target" in c.lower()]
print("Detected targets:", target_cols)

X_train = train_df.drop(columns=target_cols)
y_train = train_df[target_cols]

X_test  = test_df.copy()

print("✅ Split done")
print("X_train:", X_train.shape, "y_train:", y_train.shape, "X_test:", X_test.shape)

Detected targets: ['O3_target', 'NO2_target']
✅ Split done
X_train: (43513, 137) y_train: (43513, 2) X_test: (43248, 137)


In [9]:
sat_cols = ["NO2_satellite","HCHO_satellite","ratio_satellite"]
X_train = X_train.drop(columns=[c for c in sat_cols if c in X_train.columns], errors="ignore")
X_test  = X_test.drop(columns=[c for c in sat_cols if c in X_test.columns], errors="ignore")
print("✅ Dropped satellite columns (if present)")

✅ Dropped satellite columns (if present)


In [10]:
#CELL 9 — Scale X (fit on train only)
from sklearn.preprocessing import StandardScaler

def fit_transform_scalers(Xtr, Xte, groups: dict):
    scalers = {}
    Xtr_out = Xtr.copy()
    Xte_out = Xte.copy()
    for gname, cols in groups.items():
        cols = [c for c in cols if c in Xtr_out.columns and c in Xte_out.columns]
        if not cols:
            continue
        sc = StandardScaler()
        Xtr_out[cols] = sc.fit_transform(Xtr_out[cols])
        Xte_out[cols] = sc.transform(Xte_out[cols])
        scalers[gname] = (sc, cols)
    return Xtr_out, Xte_out, scalers

groups = {
    "time_raw":     ["year","month","day","hour"],
    "time_cyclic":  ["hour_sin","hour_cos","month_sin","month_cos","dow_sin","dow_cos"],
    "forecast_met": ["T_forecast","q_forecast","u_forecast","v_forecast","w_forecast","wind_speed_h","wind_dir_sin","wind_dir_cos"],
    "pollut_fore":  ["O3_forecast","NO2_forecast"],
    #"satellite":    ["NO2_satellite","HCHO_satellite","ratio_satellite"],
}

X_train_scaled, X_test_scaled, x_scalers = fit_transform_scalers(X_train, X_test, groups)

print("✅ X scaled")
print("X_train_scaled:", X_train_scaled.shape)

✅ X scaled
X_train_scaled: (43513, 134)


In [11]:
#CELL 10 — Residual targets (clean) + scale residual y + save scaler
import joblib

# Clean target + forecast columns (for residual stability)
needed = ["O3_target","NO2_target","O3_forecast","NO2_forecast"]
for c in needed:
    print(c, "NaNs:", train_df[c].isna().sum())

# Clean y targets
y_train = y_train.replace([np.inf, -np.inf], np.nan)
y_train = y_train.interpolate(method="time").ffill().bfill()

# Ensure forecast has no NaNs
train_df[["O3_forecast","NO2_forecast"]] = (
    train_df[["O3_forecast","NO2_forecast"]]
    .replace([np.inf, -np.inf], np.nan)
    .interpolate(method="time").ffill().bfill()
)

# Residual y = target - forecast
y_train_residual = y_train.copy()
y_train_residual["O3_target"]  = y_train["O3_target"]  - train_df["O3_forecast"]
y_train_residual["NO2_target"] = y_train["NO2_target"] - train_df["NO2_forecast"]

# Align X with residual y (drop any remaining NaNs)
mask = y_train_residual.notna().all(axis=1)
X_train_scaled = X_train_scaled.loc[mask]
y_train_residual = y_train_residual.loc[mask]

print("✅ Residual y ready")
print("Residual NaNs:", y_train_residual.isna().sum())
print("Rows kept:", len(y_train_residual))

# Scale residual targets
y_res_scaler = StandardScaler()
y_train_residual_scaled = y_train_residual.copy()
y_train_residual_scaled.loc[:, :] = y_res_scaler.fit_transform(y_train_residual.values)

os.makedirs("data", exist_ok=True)
joblib.dump(y_res_scaler, "data/y_res_scaler.pkl")

print("✅ Residual y scaled + saved scaler -> data/y_res_scaler.pkl")

O3_target NaNs: 18432
NO2_target NaNs: 18432
O3_forecast NaNs: 0
NO2_forecast NaNs: 0
✅ Residual y ready
Residual NaNs: O3_target     0
NO2_target    0
dtype: int64
Rows kept: 43513
✅ Residual y scaled + saved scaler -> data/y_res_scaler.pkl


In [12]:
#CELL 11 — Create sequences
def create_sequences(X, y, seq_length=24):
    Xs, ys = [], []
    for i in range(len(X) - seq_length):
        Xs.append(X.iloc[i:i+seq_length].values)
        ys.append(y.iloc[i+seq_length].values)
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_train_scaled, y_train_residual_scaled, seq_length=24)

print("✅ Sequences created")
print("X_seq:", X_seq.shape)
print("y_seq:", y_seq.shape)
print("Finite y_seq:", np.isfinite(y_seq).all(), "| NaNs:", np.isnan(y_seq).sum())

✅ Sequences created
X_seq: (43489, 24, 134)
y_seq: (43489, 2)
Finite y_seq: True | NaNs: 0


In [13]:
#CELL 12 — Save sequences + feature list
os.makedirs("data", exist_ok=True)

np.savez_compressed("data/sequences.npz", X_seq=X_seq, y_seq=y_seq)

feature_list = list(X_train_scaled.columns)
with open("data/feature_list.json", "w") as f:
    json.dump(feature_list, f)

print("✅ Saved:")
print(" - data/sequences.npz")
print(" - data/y_res_scaler.pkl")
print(" - data/feature_list.json")
print("Feature count:", len(feature_list))

✅ Saved:
 - data/sequences.npz
 - data/y_res_scaler.pkl
 - data/feature_list.json
Feature count: 134
